# Recommendation System with Collaborative Filtering
## CODTECH Internship - Task 4

**Objective:** Build a recommendation system using collaborative filtering and matrix factorization techniques

**Dataset:** MovieLens 100K (100,000 movie ratings)

**Approach:**
1. Data Loading and Exploration
2. Exploratory Data Analysis
3. User-Based Collaborative Filtering
4. Item-Based Collaborative Filtering
5. Matrix Factorization (SVD)
6. Model Evaluation and Comparison
7. Recommendation Generation
8. Performance Analysis

## 1. Import Required Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Recommendation system libraries
from surprise import Dataset, Reader
from surprise import KNNBasic, KNNWithMeans, SVD, SVDpp, NMF
from surprise.model_selection import train_test_split, cross_validate, GridSearchCV
from surprise import accuracy
from collections import defaultdict

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set style
plt.style.use('ggplot')
sns.set_palette('husl')

print("✅ All libraries imported successfully!")

## 2. Load and Explore Dataset

In [ ]:
# Load MovieLens 100K dataset
print("Loading MovieLens 100K dataset...")
data = Dataset.load_builtin('ml-100k')

# Get the raw ratings dataframe
df = pd.DataFrame(data.raw_ratings, columns=['user_id', 'item_id', 'rating', 'timestamp'])

print("\n✅ Dataset loaded successfully!")
print("="*60)
print("Dataset Information:")
print(f"Total ratings: {len(df):,}")
print(f"Number of users: {df['user_id'].nunique():,}")
print(f"Number of items (movies): {df['item_id'].nunique():,}")
print(f"Rating scale: {df['rating'].min()} to {df['rating'].max()}")
print(f"Average rating: {df['rating'].mean():.2f}")
print("="*60)

# Display first few rows
print("\nFirst few ratings:")
df.head(10)

In [ ]:
# Dataset statistics
print("\nDataset Statistics:")
print("="*60)
print(df.describe())

print("\n" + "="*60)
print("Data Sparsity Analysis:")
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()
num_ratings = len(df)
sparsity = (1 - (num_ratings / (num_users * num_items))) * 100

print(f"Matrix size: {num_users} users × {num_items} items = {num_users * num_items:,} cells")
print(f"Filled cells: {num_ratings:,}")
print(f"Sparsity: {sparsity:.2f}%")
print(f"Density: {100 - sparsity:.2f}%")

## 3. Exploratory Data Analysis

In [ ]:
# Rating distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Rating distribution
rating_counts = df['rating'].value_counts().sort_index()
axes[0, 0].bar(rating_counts.index, rating_counts.values, color='steelblue', alpha=0.8)
axes[0, 0].set_xlabel('Rating', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Distribution of Ratings', fontsize=14, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# Add count labels
for i, (rating, count) in enumerate(zip(rating_counts.index, rating_counts.values)):
    axes[0, 0].text(rating, count, f'{count:,}', ha='center', va='bottom', fontsize=10)

# 2. Ratings per user distribution
ratings_per_user = df.groupby('user_id').size()
axes[0, 1].hist(ratings_per_user, bins=50, color='coral', alpha=0.8, edgecolor='black')
axes[0, 1].set_xlabel('Number of Ratings', fontsize=12)
axes[0, 1].set_ylabel('Number of Users', fontsize=12)
axes[0, 1].set_title('Ratings per User Distribution', fontsize=14, fontweight='bold')
axes[0, 1].axvline(ratings_per_user.mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {ratings_per_user.mean():.1f}')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Ratings per item distribution
ratings_per_item = df.groupby('item_id').size()
axes[1, 0].hist(ratings_per_item, bins=50, color='lightgreen', alpha=0.8, edgecolor='black')
axes[1, 0].set_xlabel('Number of Ratings', fontsize=12)
axes[1, 0].set_ylabel('Number of Movies', fontsize=12)
axes[1, 0].set_title('Ratings per Movie Distribution', fontsize=14, fontweight='bold')
axes[1, 0].axvline(ratings_per_item.mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {ratings_per_item.mean():.1f}')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Average rating per movie distribution
avg_rating_per_item = df.groupby('item_id')['rating'].mean()
axes[1, 1].hist(avg_rating_per_item, bins=30, color='plum', alpha=0.8, edgecolor='black')
axes[1, 1].set_xlabel('Average Rating', fontsize=12)
axes[1, 1].set_ylabel('Number of Movies', fontsize=12)
axes[1, 1].set_title('Average Rating per Movie Distribution', fontsize=14, fontweight='bold')
axes[1, 1].axvline(avg_rating_per_item.mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {avg_rating_per_item.mean():.2f}')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Top rated movies
min_ratings = 50  # Minimum ratings required
movie_stats = df.groupby('item_id').agg({
    'rating': ['mean', 'count']
}).reset_index()
movie_stats.columns = ['item_id', 'avg_rating', 'num_ratings']
movie_stats = movie_stats[movie_stats['num_ratings'] >= min_ratings]

# Top 20 highest rated movies
top_movies = movie_stats.nlargest(20, 'avg_rating')

plt.figure(figsize=(14, 8))
bars = plt.barh(range(len(top_movies)), top_movies['avg_rating'], 
                color=plt.cm.viridis(top_movies['avg_rating']/5), alpha=0.8)
plt.yticks(range(len(top_movies)), [f"Movie {id}" for id in top_movies['item_id']])
plt.xlabel('Average Rating', fontsize=12)
plt.ylabel('Movie ID', fontsize=12)
plt.title(f'Top 20 Highest Rated Movies (min. {min_ratings} ratings)', 
         fontsize=14, fontweight='bold')
plt.xlim(3.5, 5.0)
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, rating, count) in enumerate(zip(bars, top_movies['avg_rating'], top_movies['num_ratings'])):
    plt.text(rating, bar.get_y() + bar.get_height()/2, 
            f'{rating:.2f} ({count} ratings)', 
            va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Most popular movies (by number of ratings)
most_popular = movie_stats.nlargest(20, 'num_ratings')

plt.figure(figsize=(14, 8))
bars = plt.barh(range(len(most_popular)), most_popular['num_ratings'], 
                color='coral', alpha=0.8)
plt.yticks(range(len(most_popular)), [f"Movie {id}" for id in most_popular['item_id']])
plt.xlabel('Number of Ratings', fontsize=12)
plt.ylabel('Movie ID', fontsize=12)
plt.title('Top 20 Most Popular Movies (Most Rated)', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, count, rating) in enumerate(zip(bars, most_popular['num_ratings'], most_popular['avg_rating'])):
    plt.text(count, bar.get_y() + bar.get_height()/2, 
            f'{count} (avg: {rating:.2f})', 
            va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Prepare Data for Modeling

In [ ]:
# Split data into train and test sets
print("Splitting data into train and test sets...")
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

print(f"\nTraining set size: {trainset.n_ratings:,} ratings")
print(f"Test set size: {len(testset):,} ratings")
print(f"Split ratio: 75% train, 25% test")
print("\n✅ Data split completed!")

## 5. User-Based Collaborative Filtering

In [ ]:
# User-based collaborative filtering with cosine similarity
print("Training User-Based Collaborative Filtering model...")
print("="*60)

# Configure similarity options for user-based CF
sim_options_user = {
    'name': 'cosine',
    'user_based': True  # User-based collaborative filtering
}

# Create and train the model
user_based_model = KNNWithMeans(k=40, sim_options=sim_options_user, verbose=True)
user_based_model.fit(trainset)

print("\n✅ User-Based model trained!")

In [ ]:
# Evaluate User-Based CF
print("\nEvaluating User-Based Collaborative Filtering...")
predictions_user = user_based_model.test(testset)

# Calculate metrics
rmse_user = accuracy.rmse(predictions_user, verbose=False)
mae_user = accuracy.mae(predictions_user, verbose=False)

print("\nUser-Based CF Performance:")
print("="*60)
print(f"RMSE (Root Mean Squared Error): {rmse_user:.4f}")
print(f"MAE (Mean Absolute Error): {mae_user:.4f}")
print("="*60)

## 6. Item-Based Collaborative Filtering

In [ ]:
# Item-based collaborative filtering with cosine similarity
print("Training Item-Based Collaborative Filtering model...")
print("="*60)

# Configure similarity options for item-based CF
sim_options_item = {
    'name': 'cosine',
    'user_based': False  # Item-based collaborative filtering
}

# Create and train the model
item_based_model = KNNWithMeans(k=40, sim_options=sim_options_item, verbose=True)
item_based_model.fit(trainset)

print("\n✅ Item-Based model trained!")

In [ ]:
# Evaluate Item-Based CF
print("\nEvaluating Item-Based Collaborative Filtering...")
predictions_item = item_based_model.test(testset)

# Calculate metrics
rmse_item = accuracy.rmse(predictions_item, verbose=False)
mae_item = accuracy.mae(predictions_item, verbose=False)

print("\nItem-Based CF Performance:")
print("="*60)
print(f"RMSE (Root Mean Squared Error): {rmse_item:.4f}")
print(f"MAE (Mean Absolute Error): {mae_item:.4f}")
print("="*60)

## 7. Matrix Factorization - SVD (Singular Value Decomposition)

In [ ]:
# Train SVD model
print("Training SVD (Matrix Factorization) model...")
print("="*60)

svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, verbose=True)
svd_model.fit(trainset)

print("\n✅ SVD model trained!")

In [ ]:
# Evaluate SVD
print("\nEvaluating SVD model...")
predictions_svd = svd_model.test(testset)

# Calculate metrics
rmse_svd = accuracy.rmse(predictions_svd, verbose=False)
mae_svd = accuracy.mae(predictions_svd, verbose=False)

print("\nSVD Performance:")
print("="*60)
print(f"RMSE (Root Mean Squared Error): {rmse_svd:.4f}")
print(f"MAE (Mean Absolute Error): {mae_svd:.4f}")
print("="*60)

## 8. Additional Matrix Factorization - NMF (Non-Negative Matrix Factorization)

In [ ]:
# Train NMF model
print("Training NMF (Non-Negative Matrix Factorization) model...")
print("="*60)

nmf_model = NMF(n_factors=15, n_epochs=50, verbose=True)
nmf_model.fit(trainset)

print("\n✅ NMF model trained!")

In [ ]:
# Evaluate NMF
print("\nEvaluating NMF model...")
predictions_nmf = nmf_model.test(testset)

# Calculate metrics
rmse_nmf = accuracy.rmse(predictions_nmf, verbose=False)
mae_nmf = accuracy.mae(predictions_nmf, verbose=False)

print("\nNMF Performance:")
print("="*60)
print(f"RMSE (Root Mean Squared Error): {rmse_nmf:.4f}")
print(f"MAE (Mean Absolute Error): {mae_nmf:.4f}")
print("="*60)

## 9. Model Comparison

In [ ]:
# Compare all models
models_comparison = pd.DataFrame({
    'Model': ['User-Based CF', 'Item-Based CF', 'SVD', 'NMF'],
    'RMSE': [rmse_user, rmse_item, rmse_svd, rmse_nmf],
    'MAE': [mae_user, mae_item, mae_svd, mae_nmf]
})

print("\nModel Comparison:")
print("="*60)
print(models_comparison.to_string(index=False))
print("="*60)

# Find best model
best_model_idx = models_comparison['RMSE'].idxmin()
best_model_name = models_comparison.loc[best_model_idx, 'Model']
best_rmse = models_comparison.loc[best_model_idx, 'RMSE']
best_mae = models_comparison.loc[best_model_idx, 'MAE']

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   RMSE: {best_rmse:.4f}")
print(f"   MAE: {best_mae:.4f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# RMSE comparison
bars1 = axes[0].bar(models_comparison['Model'], models_comparison['RMSE'], 
                    color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], alpha=0.8)
axes[0].set_ylabel('RMSE', fontsize=12)
axes[0].set_title('Model Comparison - RMSE (Lower is Better)', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# MAE comparison
bars2 = axes[1].bar(models_comparison['Model'], models_comparison['MAE'], 
                    color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], alpha=0.8)
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].set_title('Model Comparison - MAE (Lower is Better)', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

# Add value labels
for bar in bars2:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Prediction Analysis

In [ ]:
# Analyze prediction errors for best model (SVD)
errors = [abs(pred.est - pred.r_ui) for pred in predictions_svd]
actual_ratings = [pred.r_ui for pred in predictions_svd]
predicted_ratings = [pred.est for pred in predictions_svd]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Error distribution
axes[0].hist(errors, bins=50, color='coral', alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Prediction Error', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Prediction Errors', fontsize=14, fontweight='bold')
axes[0].axvline(np.mean(errors), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(errors):.3f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Actual vs Predicted scatter plot
axes[1].scatter(actual_ratings, predicted_ratings, alpha=0.3, s=10)
axes[1].plot([1, 5], [1, 5], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Rating', fontsize=12)
axes[1].set_ylabel('Predicted Rating', fontsize=12)
axes[1].set_title('Actual vs Predicted Ratings', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Residual plot
residuals = np.array(predicted_ratings) - np.array(actual_ratings)
axes[2].scatter(predicted_ratings, residuals, alpha=0.3, s=10)
axes[2].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[2].set_xlabel('Predicted Rating', fontsize=12)
axes[2].set_ylabel('Residual (Predicted - Actual)', fontsize=12)
axes[2].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nPrediction Error Statistics:")
print("="*60)
print(f"Mean Error: {np.mean(errors):.4f}")
print(f"Median Error: {np.median(errors):.4f}")
print(f"Min Error: {np.min(errors):.4f}")
print(f"Max Error: {np.max(errors):.4f}")
print(f"Std Error: {np.std(errors):.4f}")

## 11. Generate Recommendations

In [ ]:
def get_top_n_recommendations(predictions, n=10):
    """
    Get top-N recommendations for each user from a set of predictions
    """
    # Create a dictionary: user -> list of (item, rating) tuples
    top_n = defaultdict(list)
    
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    
    # Sort the predictions for each user and get top N
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    
    return top_n

# Generate recommendations using SVD (best model)
print("Generating top-10 recommendations for all users...")
top_n_svd = get_top_n_recommendations(predictions_svd, n=10)

print(f"\n✅ Generated recommendations for {len(top_n_svd)} users")

In [ ]:
# Display recommendations for a sample user
sample_user_id = '196'  # Sample user

print(f"\nTop 10 Recommendations for User {sample_user_id}:")
print("="*60)

if sample_user_id in top_n_svd:
    recommendations = top_n_svd[sample_user_id]
    
    for i, (movie_id, predicted_rating) in enumerate(recommendations, 1):
        print(f"{i:2d}. Movie {movie_id:4s} - Predicted Rating: {predicted_rating:.2f}")
else:
    print(f"No recommendations available for user {sample_user_id}")

# Show what the user has already rated
user_ratings = df[df['user_id'] == sample_user_id].sort_values('rating', ascending=False)
print(f"\nUser {sample_user_id}'s Previous Ratings (Top 10):")
print("="*60)
for i, row in user_ratings.head(10).iterrows():
    print(f"Movie {row['item_id']:4s} - Rating: {row['rating']:.1f}")

In [ ]:
# Function to get recommendations for any user
def get_user_recommendations(user_id, model, n=10):
    """
    Get top-N movie recommendations for a specific user
    """
    # Get all items (movies)
    all_items = df['item_id'].unique()
    
    # Get items already rated by the user
    rated_items = df[df['user_id'] == user_id]['item_id'].unique()
    
    # Get items not yet rated
    items_to_predict = [item for item in all_items if item not in rated_items]
    
    # Predict ratings for all unrated items
    predictions = []
    for item in items_to_predict:
        pred = model.predict(user_id, item)
        predictions.append((item, pred.est))
    
    # Sort by predicted rating and get top N
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_n = predictions[:n]
    
    return top_n, rated_items

# Example: Get recommendations for user '1'
user_to_recommend = '1'
recommendations, user_history = get_user_recommendations(user_to_recommend, svd_model, n=10)

print(f"\n📽️ Personalized Recommendations for User {user_to_recommend}:")
print("="*60)
print(f"User has rated {len(user_history)} movies")
print(f"\nTop 10 Recommended Movies:")
for i, (movie_id, predicted_rating) in enumerate(recommendations, 1):
    print(f"{i:2d}. Movie {movie_id:4s} - Predicted Rating: {predicted_rating:.2f} ⭐")

## 12. Interactive Recommendation System

In [ ]:
def interactive_recommendations():
    """
    Interactive function to get recommendations for any user
    """
    print("\n" + "="*80)
    print("INTERACTIVE MOVIE RECOMMENDATION SYSTEM")
    print("="*80)
    print(f"Available users: 1 to {df['user_id'].nunique()}")
    print("Type 'quit' to exit.\n")
    
    while True:
        user_input = input("Enter User ID: ")
        
        if user_input.lower() == 'quit':
            print("\nThank you for using the recommendation system!")
            break
        
        try:
            # Get recommendations
            recs, history = get_user_recommendations(user_input, svd_model, n=10)
            
            print("\n" + "-"*80)
            print(f"📊 User {user_input} Statistics:")
            print(f"   Total movies rated: {len(history)}")
            
            # Get user's rating statistics
            user_ratings = df[df['user_id'] == user_input]['rating']
            if len(user_ratings) > 0:
                print(f"   Average rating given: {user_ratings.mean():.2f}")
                print(f"   Rating range: {user_ratings.min():.1f} - {user_ratings.max():.1f}")
            
            print(f"\n🎬 Top 10 Recommended Movies:")
            print("   " + "-"*60)
            for i, (movie_id, predicted_rating) in enumerate(recs, 1):
                stars = '⭐' * int(round(predicted_rating))
                print(f"   {i:2d}. Movie {movie_id:4s} - Predicted: {predicted_rating:.2f} {stars}")
            print("-"*80 + "\n")
            
        except Exception as e:
            print(f"\nError: Could not generate recommendations. {str(e)}")
            print("Please enter a valid user ID.\n")

# Uncomment to run interactive mode
# interactive_recommendations()

## 13. Recommendation Diversity Analysis

In [ ]:
# Analyze recommendation diversity
all_recommended_items = []
for user_recs in top_n_svd.values():
    all_recommended_items.extend([item for item, _ in user_recs])

# Count frequency of recommended items
from collections import Counter
item_frequency = Counter(all_recommended_items)

print("\nRecommendation Diversity Analysis:")
print("="*60)
print(f"Total unique items recommended: {len(item_frequency)}")
print(f"Total items in dataset: {df['item_id'].nunique()}")
print(f"Coverage: {len(item_frequency)/df['item_id'].nunique()*100:.2f}%")

# Most frequently recommended items
most_common = item_frequency.most_common(20)

plt.figure(figsize=(14, 6))
items, counts = zip(*most_common)
plt.bar(range(len(items)), counts, color='steelblue', alpha=0.8)
plt.xticks(range(len(items)), [f"Movie {item}" for item in items], rotation=45, ha='right')
plt.xlabel('Movie', fontsize=12)
plt.ylabel('Recommendation Frequency', fontsize=12)
plt.title('Most Frequently Recommended Movies', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 10 Most Recommended Movies:")
print("-"*60)
for i, (item, count) in enumerate(most_common[:10], 1):
    print(f"{i:2d}. Movie {item:4s} - Recommended {count} times")

## 14. Cross-Validation

In [ ]:
# Perform 5-fold cross-validation on SVD
print("Performing 5-fold cross-validation on SVD model...")
print("This may take a few minutes...\n")

cv_results = cross_validate(svd_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

print("\n" + "="*60)
print("Cross-Validation Results:")
print("="*60)
print(f"RMSE - Mean: {cv_results['test_rmse'].mean():.4f} (+/- {cv_results['test_rmse'].std():.4f})")
print(f"MAE  - Mean: {cv_results['test_mae'].mean():.4f} (+/- {cv_results['test_mae'].std():.4f})")
print("="*60)

In [ ]:
# Visualize cross-validation results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE across folds
axes[0].plot(range(1, 6), cv_results['test_rmse'], marker='o', linewidth=2, markersize=8)
axes[0].axhline(y=cv_results['test_rmse'].mean(), color='r', linestyle='--', 
               linewidth=2, label=f"Mean: {cv_results['test_rmse'].mean():.4f}")
axes[0].set_xlabel('Fold', fontsize=12)
axes[0].set_ylabel('RMSE', fontsize=12)
axes[0].set_title('RMSE Across Cross-Validation Folds', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xticks(range(1, 6))

# MAE across folds
axes[1].plot(range(1, 6), cv_results['test_mae'], marker='o', linewidth=2, 
            markersize=8, color='coral')
axes[1].axhline(y=cv_results['test_mae'].mean(), color='r', linestyle='--', 
               linewidth=2, label=f"Mean: {cv_results['test_mae'].mean():.4f}")
axes[1].set_xlabel('Fold', fontsize=12)
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].set_title('MAE Across Cross-Validation Folds', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xticks(range(1, 6))

plt.tight_layout()
plt.show()

## 15. Final Summary

In [ ]:
# Comprehensive summary
print("\n" + "="*80)
print("RECOMMENDATION SYSTEM - FINAL SUMMARY")
print("="*80)

print("\n📊 DATASET INFORMATION:")
print(f"   • Dataset: MovieLens 100K")
print(f"   • Total Ratings: {len(df):,}")
print(f"   • Number of Users: {df['user_id'].nunique():,}")
print(f"   • Number of Movies: {df['item_id'].nunique():,}")
print(f"   • Rating Scale: {df['rating'].min():.0f} - {df['rating'].max():.0f}")
print(f"   • Average Rating: {df['rating'].mean():.2f}")
print(f"   • Data Sparsity: {sparsity:.2f}%")

print("\n🤖 MODELS IMPLEMENTED:")
print("   1. User-Based Collaborative Filtering")
print("   2. Item-Based Collaborative Filtering")
print("   3. SVD (Singular Value Decomposition)")
print("   4. NMF (Non-Negative Matrix Factorization)")

print("\n📈 MODEL PERFORMANCE:")
print(models_comparison.to_string(index=False))

print(f"\n🏆 BEST PERFORMING MODEL:")
print(f"   • Model: {best_model_name}")
print(f"   • RMSE: {best_rmse:.4f}")
print(f"   • MAE: {best_mae:.4f}")

print("\n✅ CROSS-VALIDATION RESULTS (SVD):")
print(f"   • RMSE: {cv_results['test_rmse'].mean():.4f} ± {cv_results['test_rmse'].std():.4f}")
print(f"   • MAE: {cv_results['test_mae'].mean():.4f} ± {cv_results['test_mae'].std():.4f}")

print("\n🎯 RECOMMENDATION STATISTICS:")
print(f"   • Users with recommendations: {len(top_n_svd):,}")
print(f"   • Unique movies recommended: {len(item_frequency):,}")
print(f"   • Catalog coverage: {len(item_frequency)/df['item_id'].nunique()*100:.2f}%")

print("\n💡 KEY INSIGHTS:")
print("   • Matrix factorization (SVD) typically outperforms memory-based methods")
print("   • SVD handles sparsity better than collaborative filtering")
print("   • Item-based CF often performs better than user-based CF")
print("   • Lower RMSE indicates better prediction accuracy")

print("\n🚀 RECOMMENDATIONS READY:")
print("   ✓ Personalized recommendations for all users")
print("   ✓ Top-10 items per user based on predicted ratings")
print("   ✓ Interactive recommendation function available")

print("\n" + "="*80)
print("✅ TASK COMPLETED SUCCESSFULLY!")
print("="*80)